# Laboratório — Regressão linear e mínimos quadrados

Este notebook acompanha a **Aula 04 do módulo 03 · Machine Learning clássico**.

**Problema:** prever o consumo diário de energia de um prédio a partir de variáveis disponíveis antes da operação.

Você irá:

- resolver OLS com `numpy.linalg.lstsq`;
- verificar posto, valores singulares e ortogonalidade dos resíduos;
- reproduzir a solução com `LinearRegression`;
- implementar gradiente descendente em dados padronizados;
- comparar com o baseline da média no teste temporal reservado;
- diagnosticar resíduos, outlier, multicolinearidade e extrapolação.

> Os dados são sintéticos. Os resultados demonstram mecanismos e não medem um prédio real.

## 1. Ambiente e dependências

Requisitos mínimos: Python 3.11, NumPy 2.0, pandas 2.0, Matplotlib 3.8 e scikit-learn 1.5. Não há rede, segredos ou arquivos externos.

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", version("matplotlib"))
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 2. Protocolo antes dos resultados

- **Unidade:** dia de operação do prédio.
- **Instante de predição:** início do dia.
- **Target:** consumo diário em kWh.
- **Features:** temperatura prevista, ocupação planejada e umidade prevista.
- **Split:** primeiros 80% dos dias para desenvolvimento; últimos 20% para teste temporal.
- **Baseline:** média do consumo no desenvolvimento.
- **Modelo:** regressão linear sem ajuste pelo teste.
- **Métricas:** MAE, RMSE e (R^2), sempre no mesmo conjunto.
- **Limites:** dados sintéticos, relação parcialmente não linear e nenhuma interpretação causal.

Hipóteses: o modelo superará a média fora da amostra; `lstsq` e scikit-learn produzirão previsões numericamente equivalentes; o gradiente descendente convergirá para a solução em features padronizadas.

## 3. Gerar dados com ordem temporal

O consumo contém uma parte linear, uma curvatura moderada da temperatura e ruído cuja dispersão cresce em dias mais quentes. Essa construção permite que OLS seja útil, mas deixa padrões diagnósticos realistas.

In [ ]:
n = 420
day = np.arange(n)
temperature = 23 + 7 * np.sin(2 * np.pi * day / 180) + rng.normal(0, 2.2, n)
occupancy = np.clip(0.62 + 0.18 * np.sin(2 * np.pi * day / 7) + rng.normal(0, 0.08, n), 0.12, 1.0)
humidity = np.clip(68 - 0.65 * (temperature - 23) + rng.normal(0, 5, n), 35, 92)
noise_sd = 7 + 0.28 * np.maximum(temperature - 15, 0)
noise = rng.normal(0, noise_sd)

consumption = (
    115
    + 4.2 * temperature
    + 82 * occupancy
    - 0.65 * humidity
    + 0.22 * (temperature - 24) ** 2
    + noise
)

data = pd.DataFrame({
    "day": day,
    "temperature_c": temperature,
    "occupancy": occupancy,
    "humidity_pct": humidity,
    "consumption_kwh": consumption,
})

print(data.head(3).round(3).to_string(index=False))
print("\nShape:", data.shape)
print("Faixa de temperatura:", tuple(np.round(data["temperature_c"].agg(["min", "max"]), 3)))
print("Faixa de consumo:", tuple(np.round(data["consumption_kwh"].agg(["min", "max"]), 3)))

### Verificações de entrada e split

Como o uso será em dias futuros, preservamos a ordem. O teste é reservado antes de qualquer ajuste. `day` controla tempo, mas não entra como feature nesta hipótese inicial.

In [ ]:
FEATURES = ["temperature_c", "occupancy", "humidity_pct"]
TARGET = "consumption_kwh"
cut = int(0.80 * len(data))

train = data.iloc[:cut].copy()
test = data.iloc[cut:].copy()
X_train = train[FEATURES].to_numpy()
y_train = train[TARGET].to_numpy()
X_test = test[FEATURES].to_numpy()
y_test = test[TARGET].to_numpy()

assert train["day"].max() < test["day"].min()
assert set(train["day"]).isdisjoint(set(test["day"]))
assert X_train.shape == (336, 3) and X_test.shape == (84, 3)
assert np.isfinite(X_train).all() and np.isfinite(y_train).all()
assert np.isfinite(X_test).all() and np.isfinite(y_test).all()

print("Treino:", X_train.shape, "dias", (train.day.min(), train.day.max()))
print("Teste reservado:", X_test.shape, "dias", (test.day.min(), test.day.max()))

## 4. Baseline: média do desenvolvimento

O baseline aprende uma única quantidade, a média de `y_train`, e a reutiliza no teste. Calcular a média no teste seria leakage.

In [ ]:
baseline_value = y_train.mean()
baseline_prediction = np.full_like(y_test, baseline_value, dtype=float)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, baseline_prediction)
print("Média aprendida no treino:", round(baseline_value, 6))
print("Baseline no teste:", {k: round(v, 6) for k, v in baseline_metrics.items()})

## 5. OLS com `numpy.linalg.lstsq`

Acrescentamos uma coluna de uns à matriz de projeto. A rotina retorna coeficientes, resíduos agregados, posto e valores singulares sem formar a inversa de (X^TX).

In [ ]:
X_train_design = np.column_stack([np.ones(len(X_train)), X_train])
X_test_design = np.column_stack([np.ones(len(X_test)), X_test])

beta_lstsq, residual_sums, rank, singular_values = np.linalg.lstsq(
    X_train_design, y_train, rcond=None
)
train_prediction_lstsq = X_train_design @ beta_lstsq
test_prediction_lstsq = X_test_design @ beta_lstsq
train_residual = y_train - train_prediction_lstsq

coef_table = pd.Series(
    beta_lstsq,
    index=["intercept"] + FEATURES,
    name="coeficiente",
)
print(coef_table.round(6).to_string())
print("\nPosto:", rank, "de", X_train_design.shape[1])
print("Valores singulares:", np.round(singular_values, 6))
print("Número de condição:", round(np.linalg.cond(X_train_design), 6))

### Ortogonalidade dos resíduos no treino

Na solução OLS com intercepto, (X^Te\) deve estar próximo de zero e a soma dos resíduos também. A tolerância reconhece aritmética de ponto flutuante.

In [ ]:
orthogonality = X_train_design.T @ train_residual

print("Xᵀe:", np.array2string(orthogonality, precision=10, suppress_small=True))
print("Soma dos resíduos:", f"{train_residual.sum():.12e}")
print("Máximo |Xᵀe|:", f"{np.abs(orthogonality).max():.12e}")

assert rank == X_train_design.shape[1]
assert np.abs(orthogonality).max() < 1e-7
assert abs(train_residual.sum()) < 1e-8

## 6. A mesma solução com scikit-learn

`LinearRegression` trata o intercepto separadamente. Comparamos coeficientes e previsões, não apenas uma métrica arredondada.

In [ ]:
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
test_prediction_sk = sk_model.predict(X_test)
beta_sk = np.r_[sk_model.intercept_, sk_model.coef_]

np.testing.assert_allclose(beta_sk, beta_lstsq, rtol=1e-11, atol=1e-9)
np.testing.assert_allclose(test_prediction_sk, test_prediction_lstsq, rtol=1e-11, atol=1e-9)

print("Coeficientes lstsq:", np.round(beta_lstsq, 8))
print("Coeficientes sklearn:", np.round(beta_sk, 8))
print("Maior diferença de previsão:", f"{np.max(np.abs(test_prediction_sk-test_prediction_lstsq)):.3e}")

## 7. Gradiente descendente em features padronizadas

O scaler é ajustado somente em `X_train`. Usamos a matriz padronizada com coluna de intercepto e minimizamos MSE por 4.000 passos. A loss deve diminuir e as previsões convergir para a solução OLS na mesma representação.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
Z_train = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
Z_test = np.column_stack([np.ones(len(X_test_scaled)), X_test_scaled])

beta_scaled_exact = np.linalg.lstsq(Z_train, y_train, rcond=None)[0]
beta_gd = np.zeros(Z_train.shape[1])
learning_rate = 0.05
loss_history = []

for step in range(4_000):
    error = Z_train @ beta_gd - y_train
    gradient = (2 / len(y_train)) * Z_train.T @ error
    beta_gd -= learning_rate * gradient
    if step % 20 == 0:
        loss_history.append(np.mean(error**2))

prediction_gd = Z_test @ beta_gd
prediction_scaled_exact = Z_test @ beta_scaled_exact

print("Loss inicial/final:", round(loss_history[0], 6), round(loss_history[-1], 6))
print("Norma do gradiente final:", f"{np.linalg.norm(gradient):.3e}")
print("Maior diferença GD × OLS:", f"{np.max(np.abs(prediction_gd-prediction_scaled_exact)):.3e}")

assert loss_history[-1] < loss_history[0] * 0.02
assert np.linalg.norm(gradient) < 1e-8
np.testing.assert_allclose(prediction_gd, prediction_scaled_exact, rtol=0, atol=1e-7)

### Curva de convergência

A escala vertical é logarítmica para tornar visível a queda rápida e o platô numérico.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(np.arange(len(loss_history)) * 20, loss_history, color="#2563eb")
ax.set_yscale("log")
ax.set_xlabel("Iteração")
ax.set_ylabel("MSE de treino (escala log)")
ax.set_title("Convergência do gradiente descendente")
ax.grid(alpha=0.25)
plt.show()

**Texto alternativo:** curva decrescente do MSE de treino, com queda acentuada nas primeiras iterações e estabilização ao se aproximar da solução de mínimos quadrados.

## 8. Avaliação final no teste reservado

Comparamos baseline e OLS no mesmo período. (R^2) negativo para o baseline é possível porque a média foi aprendida no período anterior e a distribuição temporal mudou.

In [ ]:
ols_metrics = regression_metrics(y_test, test_prediction_lstsq)
metrics_table = pd.DataFrame([baseline_metrics, ols_metrics], index=["Baseline média", "OLS"])
print(metrics_table.round(6).to_string())

assert ols_metrics["RMSE"] < baseline_metrics["RMSE"]
assert ols_metrics["MAE"] < baseline_metrics["MAE"]
assert ols_metrics["R2"] > 0.55

## 9. Diagnóstico de resíduos

O primeiro gráfico compara observado e previsto no teste. O segundo mostra resíduos (y-\widehat y) contra a previsão. O terceiro procura curvatura residual em temperatura.

In [ ]:
test_residual = y_test - test_prediction_lstsq

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].scatter(y_test, test_prediction_lstsq, alpha=0.7, color="#2563eb")
lims = [min(y_test.min(), test_prediction_lstsq.min()),
        max(y_test.max(), test_prediction_lstsq.max())]
axes[0].plot(lims, lims, "--", color="black", linewidth=1)
axes[0].set(xlabel="Observado (kWh)", ylabel="Previsto (kWh)", title="Observado × previsto")

axes[1].scatter(test_prediction_lstsq, test_residual, alpha=0.7, color="#dc2626")
axes[1].axhline(0, linestyle="--", color="black", linewidth=1)
axes[1].set(xlabel="Previsto (kWh)", ylabel="Resíduo (kWh)", title="Resíduo × previsto")

axes[2].scatter(X_test[:, 0], test_residual, alpha=0.7, color="#7c3aed")
axes[2].axhline(0, linestyle="--", color="black", linewidth=1)
axes[2].set(xlabel="Temperatura (°C)", ylabel="Resíduo (kWh)", title="Resíduo × temperatura")

fig.suptitle("Diagnóstico fora da amostra")
fig.tight_layout()
plt.show()

print("Média dos resíduos no teste:", round(test_residual.mean(), 6))
print("Correlação |resíduo| × temperatura:", round(np.corrcoef(np.abs(test_residual), X_test[:, 0])[0, 1], 6))

**Texto alternativo:** três dispersões mostram previsões próximas da diagonal, resíduos centrados aproximadamente em zero e a relação entre magnitude/sinal do erro e temperatura; padrões remanescentes sinalizam limites da forma linear.

## 10. Sensibilidade a um ponto extremo

Alteramos somente o target da observação de treino com maior temperatura. Esta é uma ablação didática, não uma regra de limpeza. Comparamos a inclinação de temperatura antes e depois.

In [ ]:
idx_high_leverage = np.argmax(X_train[:, 0])
y_train_outlier = y_train.copy()
y_train_outlier[idx_high_leverage] += 300

outlier_model = LinearRegression().fit(X_train, y_train_outlier)
coef_before = sk_model.coef_[0]
coef_after = outlier_model.coef_[0]

print("Temperatura do ponto alterado:", round(X_train[idx_high_leverage, 0], 6))
print("Coeficiente de temperatura original:", round(coef_before, 6))
print("Coeficiente após outlier:", round(coef_after, 6))
print("Variação relativa:", round((coef_after / coef_before - 1) * 100, 3), "%")

assert abs(coef_after - coef_before) > 0.5

## 11. Multicolinearidade exata

Acrescentamos temperatura em Fahrenheit, uma combinação linear exata de Celsius e intercepto: (F=1{,}8C+32). A matriz perde posto. Um vetor na direção nula altera coeficientes sem alterar previsões.

In [ ]:
temperature_f = 1.8 * X_train[:, 0] + 32
X_collinear = np.column_stack([X_train_design, temperature_f])
beta_collinear, _, rank_collinear, singular_collinear = np.linalg.lstsq(
    X_collinear, y_train, rcond=None
)

_, _, vh = np.linalg.svd(X_collinear, full_matrices=False)
null_direction = vh[-1]
beta_alternative = beta_collinear + 1_000 * null_direction
pred_original = X_collinear @ beta_collinear
pred_alternative = X_collinear @ beta_alternative

print("Shape:", X_collinear.shape, "posto:", rank_collinear)
print("Menor valor singular:", f"{singular_collinear[-1]:.3e}")
print("Número de condição:", f"{np.linalg.cond(X_collinear):.3e}")
print("Mudança máxima nos coeficientes:", round(np.max(np.abs(beta_alternative-beta_collinear)), 6))
print("Mudança máxima nas previsões:", f"{np.max(np.abs(pred_alternative-pred_original)):.3e}")

assert rank_collinear < X_collinear.shape[1]
assert np.max(np.abs(pred_alternative - pred_original)) < 1e-7

## 12. Extrapolação explícita

Criamos um cenário de 50 °C, fora da faixa do treino. O modelo sempre retorna um número, mas o protocolo não validou a relação nessa região.

In [ ]:
future_case = np.array([[50.0, np.median(X_train[:, 1]), np.median(X_train[:, 2])]])
future_prediction = sk_model.predict(future_case)[0]
train_temp_range = (X_train[:, 0].min(), X_train[:, 0].max())

print("Faixa de temperatura no treino:", tuple(np.round(train_temp_range, 6)))
print("Temperatura consultada:", future_case[0, 0])
print("Previsão extrapolada:", round(future_prediction, 6), "kWh")
print("Fora de suporte:", not (train_temp_range[0] <= future_case[0, 0] <= train_temp_range[1]))

assert future_case[0, 0] > train_temp_range[1]

## 13. Verificação final

As asserções confirmam o que este laboratório efetivamente pode provar: separação temporal, equivalência numérica, convergência e ganho sobre o baseline. Elas não provam causalidade nem validade fora do domínio sintético.

In [ ]:
checks = {
    "split_temporal": bool(train["day"].max() < test["day"].min()),
    "posto_completo_original": bool(rank == X_train_design.shape[1]),
    "residuos_ortogonais_no_treino": bool(np.abs(orthogonality).max() < 1e-7),
    "lstsq_igual_sklearn": bool(np.max(np.abs(test_prediction_sk-test_prediction_lstsq)) < 1e-9),
    "gd_convergiu_para_ols": bool(np.max(np.abs(prediction_gd-prediction_scaled_exact)) < 1e-7),
    "ols_superou_baseline": bool(ols_metrics["RMSE"] < baseline_metrics["RMSE"]),
    "colinearidade_detectada": bool(rank_collinear < X_collinear.shape[1]),
    "extrapolacao_sinalizada": bool(future_case[0, 0] > train_temp_range[1]),
}

assert all(checks.values())
for name, passed in checks.items():
    print(f"{name}: {'OK' if passed else 'FALHOU'}")

print("\nResumo numérico")
print("RMSE baseline:", round(baseline_metrics["RMSE"], 6))
print("RMSE OLS:", round(ols_metrics["RMSE"], 6))
print("R² OLS:", round(ols_metrics["R2"], 6))
print("Maior |Xᵀe|:", f"{np.abs(orthogonality).max():.3e}")
print("Diferença GD × OLS:", f"{np.max(np.abs(prediction_gd-prediction_scaled_exact)):.3e}")

## 14. Takeaways

1. OLS projeta (y) no espaço das colunas de (X) e minimiza a SSE.
2. `lstsq` resolve o problema sem calcular ((X^TX)^{-1}) explicitamente.
3. Com intercepto, os resíduos de treino somam aproximadamente zero e satisfazem (X^Te\approx0).
4. `LinearRegression` reproduziu a solução numérica de `lstsq`.
5. Gradiente descendente convergiu quando as features foram padronizadas somente no treino.
6. O modelo precisa superar um baseline no conjunto correto, não apenas ajustar o treino.
7. Resíduos revelam limites que MAE, RMSE e (R^2) agregados escondem.
8. Um ponto extremo pode deslocar coeficientes por causa da perda quadrática.
9. Dependência linear permite coeficientes diferentes com previsões idênticas.
10. Uma previsão fora do suporte é um número, não evidência de validade.
11. Coeficientes descrevem associações condicionais; este experimento não identifica causas.

### Próximo passo

Na Aula 05, investigue como Ridge, Lasso e Elastic Net modificam a função objetivo para estabilizar coeficientes e controlar complexidade. A escolha da penalização deverá ocorrer sem consultar o teste.

## 15. Exercício de transferência

Escolha um target contínuo do seu domínio e entregue:

- contrato de predição e baseline;
- split coerente com tempo ou entidades;
- solução por `lstsq` e scikit-learn;
- shapes, posto, valores singulares e número de condição;
- MAE/RMSE no teste reservado;
- três gráficos de resíduos;
- uma contraprova com outlier ou colinearidade;
- alerta automático de extrapolação;
- conclusão que separe previsão, associação e causalidade.